In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
uploaded = files.upload()  # Upload prueba.xlsx manually when prompted

Saving prueba.xlsx to prueba (1).xlsx


In [5]:
R = pd.read_excel("prueba.xlsx", sheet_name="Matriz de Simulacion").values  # [30000 x 55]
P1 = pd.read_excel("prueba.xlsx", sheet_name="P1").values.flatten()         # [55,]
P2 = pd.read_excel("prueba.xlsx", sheet_name="P2").values.flatten()         # [55,]

print(f"R shape: {R.shape}")
print(f"P1 shape: {P1.shape}, sum: {P1.sum():.4f}")
print(f"P2 shape: {P2.shape}, sum: {P2.sum():.4f}")

R shape: (30000, 55)
P1 shape: (55,), sum: 1.0000
P2 shape: (55,), sum: 1.0000


In [6]:
def portfolio_stats(R, P, name="Portfolio"):
    """
    Computes expected return and volatility for a portfolio.

    Parameters:
        R : np.ndarray [n_simulations x n_assets] — simulated annual returns
        P : np.ndarray [n_assets,] — asset weights vector
        name : str — portfolio label for display

    Returns:
        dict with expected_return and volatility
    """
    portfolio_returns = R @ P                  # [30000,] weighted return per simulation
    expected_return = np.mean(portfolio_returns)
    volatility = np.std(portfolio_returns, ddof=1)  # ddof=1 for sample std

    return {
        "Portfolio": name,
        "Expected Return": round(expected_return, 6),
        "Volatility": round(volatility, 6)
    }

In [7]:
results = pd.DataFrame([
    portfolio_stats(R, P1, name="P1"),
    portfolio_stats(R, P2, name="P2")
])

results["Expected Return (%)"] = (results["Expected Return"] * 100).round(4)
results["Volatility (%)"] = (results["Volatility"] * 100).round(4)

print(results[["Portfolio", "Expected Return (%)", "Volatility (%)"]].to_string(index=False))

Portfolio  Expected Return (%)  Volatility (%)
       P1               9.1191         20.6253
       P2               5.8419         10.1071


## Una nota personal para para nuestro cliente inversionista

Antes de ver los números, vale la pena entender qué significan en la práctica.

Simulamos 30,000 escenarios posibles de retorno anual para dos portafolios
distintos, P1 y P2, cada uno con una estrategia de inversión diferente.
El resultado no es una predicción, es un mapa de probabilidades:
nos dice qué tan lejos puede llegar cada portafolio, y a qué costo.

**P1 — El portafolio agresivo**
Retorno esperado de 9.12% anual, pero con una volatilidad de 20.63%.
Eso significa que en un año malo, este portafolio puede caer
considerablemente antes de recuperarse. Es una apuesta por el crecimiento
de largo plazo. Si usted puede dormir tranquilo viendo su inversión
bajar un 20% en pantalla sabiendo que históricamente se recupera,
P1 es su portafolio.

**P2 — El portafolio conservador**
Retorno esperado de 5.84% anual con una volatilidad de 10.11%.
Menos retorno, pero el camino es mucho más estable.
Lo interesante: por cada punto de riesgo que asume,
P2 genera más retorno relativo que P1. Es el portafolio
para quien prioriza preservar capital sin renunciar al crecimiento.

**¿Cuál elegir?**
No existe la respuesta correcta universal. Existe la respuesta
correcta para su perfil, su horizonte de inversión y su tolerancia
al riesgo. Nuestro trabajo como asesores es ayudarle a encontrarla.

## Resultados e Interpretación

### 2.1 Formulación matemática

Dado R de dimensión [30,000 x 55] y un vector de pesos P de dimensión [55,],
el retorno y la volatilidad esperados del portafolio se calculan así:

**Paso 1 — Retornos simulados del portafolio:**

    r_p = R · P^T   →   vector [30,000 x 1]

Cada elemento representa el retorno ponderado del portafolio en una simulación.

**Paso 2 — Retorno esperado:**

    E[R_p] = mean(r_p)

**Paso 3 — Volatilidad esperada:**

    σ_p = std(r_p)   [desviación estándar muestral, ddof=1]

Este enfoque es equivalente a la fórmula clásica σ_p = sqrt(P · Σ · P^T),
donde Σ es la matriz de covarianzas de R. Con 30,000 simulaciones,
la estimación es estadísticamente robusta.

---

### 2.2 Resultados numéricos

| Portfolio | Retorno Esperado | Volatilidad |
|-----------|-----------------|-------------|
| P1        | 9.12%           | 20.63%      |
| P2        | 5.84%           | 10.11%      |

**Lectura de los resultados:**

- P1 ofrece el mayor retorno absoluto (9.12%), pero asume casi el doble
  de volatilidad que P2 (20.63% vs 10.11%). Perfil de mayor riesgo.

- P2 es más conservador: menor retorno, pero mejor relación
  retorno/riesgo (0.58 vs 0.44 de P1). Por cada unidad de riesgo asumida,
  P2 genera más retorno relativo.

- La elección entre P1 y P2 depende del perfil de riesgo del inversionista.
  Un cliente con tolerancia alta al riesgo preferiría P1; uno conservador, P2.

## 2.3 Comparación de modelos: Claude vs Gemini

**Pregunta formulada:**
*"¿Cómo afecta una subida de tasas de la Fed al retorno y volatilidad
esperados de un portafolio de renta fija y uno de renta variable?"*

> Nota metodológica: ambos modelos recibieron la consulta en frío,
> sin contexto previo ni prompt estructurado. Esto limita el potencial
> real de cada modelo, pero permite una comparación bajo las mismas
> condiciones de partida.

---

### Tabla comparativa

| Criterio | Claude | Gemini |
|----------|--------|--------|
| **Profundidad** | Explicación clara del mecanismo económico, diferencia corto y mediano plazo, introduce duración y rotación sectorial. Sintético. | Análisis más amplio: mecanismo de precios, reinversión, descuento de flujos, competencia bonos-acciones, expectativas de mercado y diferencias sectoriales. |
| **Precisión** | Correcto. Relación inversa precio-tasa y efecto sobre valuaciones bien explicados. Sin errores conceptuales. | Preciso y con matices adicionales: efecto "priced in" de expectativas e impacto diferencial por sector. |
| **Tono** | Profesional y conciso. Estilo ejecutivo, orientado a briefing financiero. | Didáctico y conversacional. Más accesible, ligeramente menos formal. |
| **Estructura** | Muy ordenado. Secciones claras y tabla resumen. Alta legibilidad. | Bien estructurado pero más extenso. Subtítulos y secciones adicionales aportan detalle a costa de concisión. |

---

### Conclusión

Ambos modelos son correctos desde el punto de vista financiero y cubren
los mecanismos principales. La diferencia está en el nivel de detalle.

Gemini incorpora elementos que en un contexto de inversión real marcan
la diferencia: el rol de las expectativas del mercado frente a las
decisiones de la Fed, la competencia entre bonos y acciones por la prima
de riesgo, y los efectos heterogéneos por sector dentro de la renta variable.

Para análisis financiero y de inversión, **escogería Gemini** como punto
de partida cuando se necesita contexto amplio y profundidad de mercado.

Sin embargo, esta comparación tiene un matiz importante: Claude responde
mejor cuando se le da contexto preciso y una tarea concreta. En ese escenario,
la brecha se reduce considerablemente. Para tareas estructuradas de análisis,
automatización o generación de código, Claude es mi herramienta preferida.

En la práctica, no son modelos excluyentes. Son complementarios.